In [1]:
from IPython.display import Javascript

RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))

const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  let chunks = []

  recorder.ondataavailable = e => chunks.push(e.data)

  recorder.start()
  await sleep(time)

  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }

  recorder.stop()
})
"""

In [11]:
from IPython.display import Audio, display
from google.colab import output
from base64 import b64decode

def record(sec=5):
    display(Javascript(RECORD))

    js_result = output.eval_js(f'record({sec * 1000})')

    audio = b64decode(js_result.split(',')[1])

    file_name = 'audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)

    return file_name

print("Gravando...")
audio_file = record(5)

display(Audio(audio_file, autoplay=True))

Gravando...


<IPython.core.display.Javascript object>

In [4]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.3 MB/s eta 0:00:00


In [12]:
import whisper

model = whisper.load_model("small")
result = model.transcribe(audio_file, fp16=False, language='pt')
transcription = result["text"]
print(transcription)

 Defina o chat de IPT em três palavras.


In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

client = OpenAI(api_key="SUA-KEY")

response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[ { "role": "user", "content": transcription } ]
)

chatgpt_response = response.choices[0].message.content

print(chatgpt_response)

In [ ]:
!pip install gTTs

In [ ]:
from gtts import gTTs

gtts_object = gTTs(text=chatgpt_response, lang="pt", slow=False)
response_audio = "response.mp3"
gtts_object.save(response_audio)
display(Audio(response_audio, autoplay=True))